description, severity, certainty, response, geometry[coordinates] -> outage or no outage

In [19]:
import pandas as pd
import numpy as np
import xgboost as xgb



# Prepare the features and target variable
X = data[['event_type', 'severity', 'certainty', 'response', 'geometry_coordinates']]
y = data['outage']

# Create the DMatrix for XGBoost
dtrain = xgb.DMatrix(X, label=y)

# Set parameters for XGBoost
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
}

# Train the model
model = xgb.train(params, dtrain)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:event_type: object, severity: object, certainty: object, response: object, geometry_coordinates: object

In [20]:
data = pd.read_csv('fake_weather_outage_data.csv')
val = pd.read_csv('fake_weather_outage_val.csv')

In [11]:
!pip install xgboost

In [27]:
import ast
import numpy as np

def get_centroid(coord_str):
    # Convert string representation of list to actual list
    coords = ast.literal_eval(coord_str)
    # coords is [[[lng, lat], [lng, lat], ...]] — extract inner list
    points = coords[0]
    # Compute average longitude and latitude
    lngs, lats = zip(*points)
    return [np.mean(lngs), np.mean(lats)]

# Apply to dataframe column to replace polygon with centroid
data['geometry_centroid'] = data['geometry_coordinates'].apply(get_centroid)

In [31]:
data['geometry_centroid'].dtype

dtype('O')

In [32]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

# Prepare features and target
X = data[['event_type', 'severity', 'certainty', 'response']]#, 'geometry_centroid']]
y = data['outage']

# Simple label encoding for categorical features
for col in ['event_type', 'certainty', 'response', 'severity']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# For geometry_coordinates, either:
# - Extract numeric features, or
# - Drop the column if not usable
#X = X.drop(columns=['geometry_coordinates'])

# Create DMatrix
dtrain = xgb.DMatrix(X, label=y)

# Parameters
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
}

# Train model
model = xgb.train(params, dtrain)


C:\Users\nia_4\AppData\Local\Temp\ipykernel_7076\2818964690.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(X[col].astype(str))
C:\Users\nia_4\AppData\Local\Temp\ipykernel_7076\2818964690.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(X[col].astype(str))
C:\Users\nia_4\AppData\Local\Temp\ipykernel_7076\2818964690.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

In [37]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load data
data = pd.read_csv('fake_weather_outage_data.csv')
val = pd.read_csv('fake_weather_outage_val.csv')

# Define features without geometry
features = ['event_type', 'severity', 'certainty', 'response']
X_train = data[features]
y_train = data['outage']

X_val = val[features]
y_val = val['outage']

# Label encode categorical features
label_encoders = {}
for col in features:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_val[col] = le.transform(X_val[col].astype(str))

# Train Random Forest
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)

# Predict and evaluate
y_pred = rf.predict(X_val)

print(f"Validation Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(classification_report(y_val, y_pred))


Validation Accuracy: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



C:\Users\nia_4\AppData\Local\Temp\ipykernel_7076\3287776520.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[col] = le.fit_transform(X_train[col].astype(str))
C:\Users\nia_4\AppData\Local\Temp\ipykernel_7076\3287776520.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_val[col] = le.transform(X_val[col].astype(str))
C:\Users\nia_4\AppData\Local\Temp\ipykernel_7076\3287776520.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_

ValueError: y contains previously unseen labels: 'Severe Thunderstorm Warning'

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# Load data
data = pd.read_csv('fake_weather_outage_data.csv')
val = pd.read_csv('fake_weather_outage_val.csv')

# Prepare features and target
X = data[['event_type', 'severity', 'certainty', 'response', 'geometry_centroid']]
y = data['outage']

# Simple label encoding for categorical features
for col in ['event_type', 'certainty', 'response']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# If 'geometry_centroid' contains lists, split into two columns
if isinstance(X['geometry_centroid'].iloc[0], list):
    X[['centroid_lng', 'centroid_lat']] = pd.DataFrame(X['geometry_centroid'].tolist(), index=X.index)
    X = X.drop(columns=['geometry_centroid'])

# Instantiate and train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

# Optionally, predict on validation set (prepare val X similarly)


ModuleNotFoundError: No module named 'sklearn'

unrelated and only to restructure ad simplify data

In [41]:
import csv
import re
import pandas as pd

def extract_points(geometry_str):
    """
    Extract (x, y, z) tuples from a POLYGON Z string.
    """
    coords = re.findall(r"([-0-9\.]+) ([-0-9\.]+) ([-0-9\.]+)", geometry_str)
    return [(float(x), float(y), float(z)) for x, y, z in coords]

def process_row(row):
    geometry = row['geometry']
    points = extract_points(geometry)

    if not points:
        return None

    xs, ys, zs = zip(*points)

    avg_x = sum(xs) / len(xs)
    avg_y = sum(ys) / len(ys)
    max_z = max(zs)
    mean_z = sum(zs) / len(zs)

    return {
        'tileid': row['tileid'],
        'center_x': avg_x,
        'center_y': avg_y,
        'max_elevation': max_z,
        'mean_elevation': mean_z
    }

# Load your CSV
df = pd.read_csv("C:/Users/nia_4/Downloads/output_file.csv")

cleaned_data = []

for _, row in df.iterrows():
    result = process_row(row)
    if result:
        cleaned_data.append(result)

# Create new simplified DataFrame
simple_df = pd.DataFrame(cleaned_data)

# Optionally convert from Web Mercator (EPSG:3857) to lat/lon (EPSG:4326)
# Assuming your data is in meters (like -105xxxxx), use pyproj
from pyproj import Transformer

transformer = Transformer.from_crs("epsg:3857", "epsg:4326", always_xy=True)

simple_df[['lon', 'lat']] = simple_df.apply(
    lambda row: pd.Series(transformer.transform(row['center_x'], row['center_y'])),
    axis=1
)

# Drop Web Mercator columns if you want
simple_df = simple_df[['tileid', 'lat', 'lon', 'max_elevation', 'mean_elevation']]

# Save to a clean CSV
simple_df.to_csv("C:/Users/nia_4/Downloads/clean_topography_data.csv", index=False)


In [44]:
# Read in the CSV file and filter for rows where 'state_id' is 'TX'
data = pd.read_csv("C:/Users/nia_4/Downloads/simplemaps_uscities_basicv1.91/uscities.csv")

filtered_data = data[data['state_id'] == 'TX']

filtered_data.to_csv("C:/Users/nia_4/Downloads/simplemaps_uscities_basicv1.91/uscities_filtered.csv", index=False)